# Chapter 13 — Microstructure and Execution

Ch12 closed the cost cycle on the QQQ closing-window MR strategy with an **after-cost Sharpe of −2.97** at the cost-aware `k_σ*` = 2.20. The spread + commission + impact wedge — about **1.62 bp round-trip** at a $30k notional trade — was several times the per-trade mean PnL (≈ 0.46 bp at `k_σ` = 2.20). The strategy is unprofitable at every threshold; capacity *Q\** = $0.

Ch12 raised one specific rescue hypothesis and closed without testing it: posting **passive limit orders** on entry instead of market orders. A market order *pays* the half-spread; a passive limit, when it fills, *earns* the half-spread. On QQQ that's a swing of about 1.448 bp round-trip — almost the entire cost wedge. If passive entries can be made to fill reliably, the strategy might survive.

This chapter tests that hypothesis honestly. The expected punchline is the canonical microstructure result: **adverse selection eats the spread credit**. Passive limits fill exactly when the market is moving against the posted side, which is precisely when filling is *worst* for the strategy. The chapter measures whether that happens here.

## Three things this chapter covers

1. **Order types are policy choices.** §2.
2. **Passive fills are adversely selected.** §3-§4.
3. **Latency is an implicit cost.** §5.

## §1 — The rescue hypothesis

The Ch12 §2 reframing: Roll's half-spread on QQQ 1-min closes is `s ≈ 0.724 bp`, and Roll's is derived from the *same* lag-1 negative autocovariance that drives Ch7's MR signal. A material chunk of the −0.028 lag-1 ρ in Ch7 is bid-ask bounce, not economic mean-reversion. The strategy was, in part, trying to fade its own counterparty's spread — a structural mismatch, because the strategy must pay the spread to capture the bounce.

The rescue hypothesis flips the sign on the structural mismatch:

- **Buy signal** → post a buy limit at `signal_close − s`. If the limit fills, the entry price is below the signal close by exactly the half-spread; the strategy *receives* `s` instead of paying it.
- **Sell signal** → post a sell limit at `signal_close + s`. Symmetric.

If every signal filled, the round-trip swing would be **+1.448 bp** — almost identical in magnitude to the Ch12 cost wedge that killed the strategy. Combined with rebate income on the maker leg, the rescue looks like it should comfortably flip the sign of the Sharpe.

The chapter tests whether it does.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CACHE = Path('../06-bridge-to-intraday/data')
qqq = pd.read_parquet(CACHE / 'qqq_1min.parquet')
# Localize to NY time; carve out RTH minutes (9:30-15:59 inclusive of 9:30, exclusive of 16:00)
qqq.index = pd.to_datetime(qqq.index, utc=True).tz_convert('America/New_York')
qqq['minute_of_day'] = qqq.index.hour * 60 + qqq.index.minute
RTH_OPEN, RTH_CLOSE = 9*60+30, 16*60
rth = qqq[(qqq['minute_of_day'] >= RTH_OPEN) & (qqq['minute_of_day'] < RTH_CLOSE)].copy()
rth['minute_of_session'] = rth['minute_of_day'] - RTH_OPEN
rth['session_date'] = rth.index.normalize()
# Per-bar log return; drop the first bar of each session (overnight gap)
rth['log_ret'] = np.log(rth['close'] / rth['close'].shift(1))
sf = rth.groupby('session_date').head(1).index
rth.loc[sf, 'log_ret'] = np.nan
print(f'Sessions: {rth["session_date"].nunique()}  Bars: {len(rth):,}')

# Ch12 handoff: Roll's half-spread = 0.724 bp = 7.24e-5 in log-return units
ROLL_S = 7.24e-5
K_SIGMA = 2.20  # Ch12 cost-aware k*
N, K = 1, 3
print(f'ROLL_S = {ROLL_S*1e4:.3f} bp   K_SIGMA = {K_SIGMA}   N={N}  K={K}')

Sessions: 251  Bars: 95,318
ROLL_S = 0.724 bp   K_SIGMA = 2.2   N=1  K=3


## §2 — Order types: a short tour

| Order type | What it does | Maker/taker | Fill certainty |
|---|---|---|---|
| **Market** | Buy/sell now at the best available price. | Taker | Near-guaranteed |
| **Limit** | Buy at `≤ p`; sell at `≥ p`. | Maker (if posted away from the touch) | Conditional |
| **Marketable limit** | A limit with price already at-or-through the touch. | Taker | Near-guaranteed |
| **IOC** | Fill what's available right now; cancel the rest. | Maker on filled portion. | Partial |
| **FOK** | All-or-nothing instant fill. | Maker if filled. | Binary |
| **Hidden** | Limit price + size not displayed. | Maker. | Conditional |
| **Pegged** | Limit price tracks the bid/offer at some offset. | Maker. | Conditional |

### Maker / taker economics

Exchanges charge takers (≈ $0.0030/share) and pay makers (≈ $0.0020-$0.0030/share). On QQQ at ≈ $695, the maker rebate is **0.043 bp/leg** — small but non-zero.

### Payment for order flow (PFOF)

A retail "market order" through Schwab/Robinhood is typically routed to a wholesale market maker (Citadel, Virtu) who fills it at-or-better than NBBO in exchange for paying the broker for the flow. The price improvement is real but smaller than the spread the wholesaler captures.

## §3 — The limit-order simulation

**Fill rule.** A passive entry posted at the signal bar's close fills if a *subsequent* bar within the K-bar hold window prints at-or-through the limit:

- **Buy limit** at `signal_close − s` fills the first time any later bar's `low ≤ signal_close − s`.
- **Sell limit** at `signal_close + s` fills the first time any later bar's `high ≥ signal_close + s`.

If neither condition is met within K bars, the limit *expires unfilled*. Unfilled signals are recorded along with the K-bar post-signal drift in strategy direction — they are the central data for the §4 toxicity diagnostic.

**Exit rule.** Unchanged from Ch12: market-on-open at bar `entry + K`. The exit is still a market order — it pays the half-spread, commission, and impact.

**Limit-price choice — at-the-touch.** Posting at `signal_close − s` means the buy limit sits *at* the implied bid (the favorable side of the touch). This is the most aggressive passive entry; Exercise 1 explores variants.

In [2]:
# Pre-compute per-session signal-window prior returns for the rolling σ estimate.
# The signal window is minutes 360-389 of session (Ch7 closing window, last 30 min).
def precompute_priors(rth_df):
    priors = {}
    dates = sorted(rth_df['session_date'].unique())
    for date in dates:
        day = rth_df[rth_df['session_date'] == date].sort_index().reset_index()
        pts = []
        for i in range(N, len(day)):
            if 360 <= day.loc[i, 'minute_of_session'] < 390 - K:
                pts.append(np.log(day.loc[i, 'close'] / day.loc[i-N, 'close']))
        priors[date] = np.array(pts)
    return priors, dates

priors_by_session, dates_sorted = precompute_priors(rth)

def backtest(rth_df, entry_mode='market', limit_offset=ROLL_S, lookback=20):
    """Event-driven backtest with market or limit entries; K-bar hold.
    Returns a trade ledger with: filled flag, pnl_log, and (for unfilled-limit
    rows) drift_K_unconditional — the post-signal drift in strategy direction.
    """
    trades = []
    for date_idx, date in enumerate(dates_sorted):
        if date_idx < lookback:  # need σ history
            continue
        # Trailing 20-session pool of prior N-bar returns → today's σ threshold
        history = np.concatenate([priors_by_session[d]
                                  for d in dates_sorted[date_idx-lookback:date_idx]])
        thresh = K_SIGMA * history.std() if len(history) > 0 else None
        if thresh is None:
            continue
        day = rth_df[rth_df['session_date'] == date].sort_index().reset_index()
        in_pos, entry_i, entry_px = 0, None, None
        pending_limit, pending_sig, pending_signal_i = None, 0, None
        for i in range(len(day)):
            mos = day.loc[i, 'minute_of_session']
            # Pending limit: check this bar's H/L for a touch
            if pending_limit is not None and i > pending_signal_i:
                bar_high = day.loc[i, 'high']
                bar_low = day.loc[i, 'low']
                if pending_sig == +1 and bar_low <= pending_limit:
                    in_pos, entry_i, entry_px = +1, i, pending_limit
                    pending_limit = None
                elif pending_sig == -1 and bar_high >= pending_limit:
                    in_pos, entry_i, entry_px = -1, i, pending_limit
                    pending_limit = None
                elif i - pending_signal_i >= K:
                    # Limit expired — record an unfilled row with the K-bar drift
                    # in strategy direction for the toxicity diagnostic.
                    sig_i = pending_signal_i
                    end_i = min(sig_i + K, len(day) - 1)
                    drift_uncond = np.log(day.loc[end_i, 'close'] /
                                          day.loc[sig_i, 'close']) * pending_sig
                    trades.append({'date': date, 'filled': False, 'pnl_log': 0.0,
                                   'drift_K_unconditional': drift_uncond})
                    pending_limit, pending_sig, pending_signal_i = None, 0, None
            # Time-stop exit at K bars after entry (market on next bar's open)
            if in_pos != 0 and (i - entry_i) >= K:
                exit_i = min(i + 1, len(day) - 1)
                exit_px = day.loc[exit_i, 'open']
                pnl = in_pos * np.log(exit_px / entry_px)
                trades.append({'date': date, 'filled': True, 'pnl_log': pnl})
                in_pos, entry_i, entry_px = 0, None, None
            # Signal detection (Ch7 window, in_pos and pending checks)
            if in_pos == 0 and pending_limit is None and \
               360 <= mos < 390 - K and i >= N:
                prior_N = np.log(day.loc[i, 'close'] / day.loc[i-N, 'close'])
                if abs(prior_N) > thresh:
                    # MR: buy after downward overshoot, sell after upward
                    sig = -1 if prior_N > 0 else +1
                    if entry_mode == 'market':
                        enxt = min(i + 1, len(day) - 1)
                        in_pos, entry_i, entry_px = sig, i, day.loc[enxt, 'open']
                    else:
                        # Buy below signal close; sell above
                        pending_limit = day.loc[i, 'close'] - sig * limit_offset
                        pending_sig, pending_signal_i = sig, i
    return pd.DataFrame(trades)

In [3]:
# Market-order baseline (Ch12 cost-aware k*, pre-cost)
tr_market = backtest(rth, entry_mode='market')
# Passive limit at-the-touch
tr_limit = backtest(rth, entry_mode='limit', limit_offset=ROLL_S)
filled = tr_limit[tr_limit['filled']].copy()
unfilled = tr_limit[~tr_limit['filled']].copy()
fill_rate = len(filled) / max(len(tr_limit), 1)

def annualize_sharpe(pnl_series, n_trades, n_sessions):
    if n_trades < 2 or pnl_series.std() == 0:
        return float('nan')
    tpy = n_trades / max(n_sessions, 1) * 252  # trades/year scaling
    return (pnl_series.mean() / pnl_series.std()) * np.sqrt(tpy)

sessions_total = tr_market['date'].nunique()
sessions_filled = filled['date'].nunique()
s_market = annualize_sharpe(tr_market['pnl_log'], len(tr_market), sessions_total)
s_limit_filled = annualize_sharpe(filled['pnl_log'], len(filled), sessions_filled)

print(f'Market signals:  {len(tr_market):3d}  mean={tr_market["pnl_log"].mean()*1e4:+.3f} bp  Sharpe={s_market:+.3f}')
print(f'Limit signals:   {len(tr_limit):3d}  filled={len(filled)}  unfilled={len(unfilled)}  fill_rate={fill_rate:.3f}')
print(f'  Filled-only:   mean={filled["pnl_log"].mean()*1e4:+.3f} bp  Sharpe={s_limit_filled:+.3f}  (pre-toxicity)')

Market signals:  200  mean=+0.461 bp  Sharpe=+1.183
Limit signals:   176  filled=157  unfilled=19  fill_rate=0.892
  Filled-only:   mean=+0.288 bp  Sharpe=+0.583  (pre-toxicity)


**Empirical result:** Fill rate **89.2%** (157 / 176). Filled-only pre-cost Sharpe **+0.583** — a +3.5-point swing from Ch12's market-order after-cost Sharpe of −2.97. **The rescue looks like it works.** §4 explains why this number is a lie.

## §4 — The adverse-selection diagnostic

A passive buy limit posted at the bid fills when sellers cross the spread to hit the bid. Some of those sellers are uninformed; some are *informed* — they're selling because they know something the passive buyer doesn't. The passive buyer just bought the top of a move down.

This is **adverse selection**. Fills are not a random sample of signals — fills are conditioned on a counterparty being willing to *cross* the spread at that exact moment, and that crossing decision is itself information.

> **Formula — toxicity diagnostic.**
>
> toxicity = drift<sub>unconditional</sub> − drift<sub>filled</sub>
>
> where:
> - drift<sub>unconditional</sub> = mean over all signal bars of the K-bar log-return in strategy direction (= mean PnL of the matched market-order backtest)
> - drift<sub>filled</sub> = mean over filled passive trades of the K-bar log-return in strategy direction
> - both in log-return units; ×1e4 for bp
>
> **Sign convention.** Positive toxicity → filled trades did *worse* than the unconditional signal set.

In [4]:
drift_unc_bp = tr_market['pnl_log'].mean() * 1e4
drift_filled_bp = filled['pnl_log'].mean() * 1e4
toxicity_bp = drift_unc_bp - drift_filled_bp
drift_unfilled_bp = unfilled['drift_K_unconditional'].mean() * 1e4

print(f'drift_unconditional (all market-order signals): {drift_unc_bp:+.4f} bp')
print(f'drift_filled        (limit fills only):         {drift_filled_bp:+.4f} bp')
print(f'toxicity = unc − filled:                        {toxicity_bp:+.4f} bp')
print(f'drift on UNFILLED signals (strategy dir):       {drift_unfilled_bp:+.4f} bp')

# Cost stack from Ch12: half-spread = 0.724 bp/leg; commission = 0.072 bp/leg;
# impact (consolidated ADV at $30k) = 0.014 bp/leg. Entry leg pays NO spread (it IS
# the passive fill). Exit leg crosses at next bar's open → pays one half-spread + comm + impact.
HALF_SPREAD_BP = 0.724
COMM_RT_BP = 0.144  # 2 × 0.072
IMPACT_RT_BP = 0.029  # 2 × 0.014 at consolidated ADV (Ch12)
exit_leg_cost_bp = HALF_SPREAD_BP + IMPACT_RT_BP/2 + COMM_RT_BP/2
# (Entry-leg commission still applies; we charge it on the entry side too.)
filled['pnl_net_log'] = filled['pnl_log'] - (exit_leg_cost_bp + COMM_RT_BP/2) / 1e4
s_limit_post = annualize_sharpe(filled['pnl_net_log'], len(filled), sessions_filled)
print(f'\nSharpe (limit, post-toxicity, net of exit spread + commission): {s_limit_post:+.3f}')

drift_unconditional (all market-order signals): +0.4612 bp
drift_filled        (limit fills only):         +0.2882 bp
toxicity = unc − filled:                        +0.1730 bp
drift on UNFILLED signals (strategy dir):       +8.1060 bp

Sharpe (limit, post-toxicity, net of exit spread + commission): -1.202


**Two findings.**

1. **Toxicity is small (+0.17 bp).** The filled subset is only slightly worse than the unconditional universe.
2. **The unfilled-bucket drift is enormous (+8.1 bp).** The 19 signals that didn't fill had over 8 bp of drift in strategy direction. The biggest winners self-cancelled.

**Re-priced with the exit-leg cost stack:** the +0.58 pre-toxicity Sharpe drops to **−1.20** — the rescue narrowed the gap from −2.97, but the exit leg still pays the wedge.

## §5 — Latency

> **Definition — signal-to-fill latency.** Wall-clock time between the signal logic emitting an order and the exchange matching engine processing it.

| Setup | Typical latency |
|---|---|
| Co-located HFT | < 1 ms |
| Cloud-hosted retail bot | 5–50 ms |
| Retail desktop on home internet | 50–500 ms |

> **Formula — random-walk latency drift.**
>
> σ<sub>drift</sub> = σ<sub>1min</sub> · √(Δt / 60 s)
>
> where:
> - σ<sub>1min</sub> = per-minute log-return standard deviation
> - Δt = latency in seconds; 60 s = reference bar duration
> - E\|drift\| under normal model ≈ 0.80 σ<sub>drift</sub>

Critically, σ<sub>drift</sub> in *absolute* bp depends only on latency. What changes with bar resolution is the *ratio* to the bar's own σ.

In [5]:
sigma_per_min_bp = rth['log_ret'].std() * 1e4  # QQQ per-bar (1-min) σ in bp
print(f'QQQ 1-min σ: {sigma_per_min_bp:.3f} bp')

rows = []
for bar_s, lat_ms in [(60, 100), (60, 500), (15, 200), (5, 200), (1, 200)]:
    # σ over the latency window — note: depends only on lat_ms, not bar_s
    sigma_drift_bp = sigma_per_min_bp * np.sqrt((lat_ms/1000) / 60)
    e_abs_drift_bp = sigma_drift_bp * np.sqrt(2 / np.pi)  # E|N(0,σ)| = σ√(2/π)
    sigma_bar_bp = sigma_per_min_bp * np.sqrt(bar_s / 60)  # σ at this bar resolution
    rows.append({'bar': f'{bar_s}s', 'latency_ms': lat_ms,
                 'σ_drift_bp': round(sigma_drift_bp, 4),
                 'E|drift|_bp': round(e_abs_drift_bp, 4),
                 'drift_σ / bar_σ': round(sigma_drift_bp / sigma_bar_bp, 3)})
lat_table = pd.DataFrame(rows)
print(lat_table.to_string(index=False))

QQQ 1-min σ: 4.312 bp
bar  latency_ms  σ_drift_bp  E|drift|_bp  drift_σ / bar_σ
60s         100      0.1760       0.1405            0.041
60s         500      0.3936       0.3141            0.091
15s         200      0.2489       0.1986            0.115
 5s         200      0.2489       0.1986            0.200
 1s         200      0.2489       0.1986            0.447


**At 1-min/100ms latency drift is 4% of bar σ — invisible. At 1-sec/200ms it's 45% — dominant.** The Ch6/Ch8 1-min bar choice was implicitly a latency-budget choice.

## §6 — Maker rebates and the realistic verdict

NASDAQ tier-1 maker rebate ≈ **$0.0030/share**. At QQQ ≈ $694.93, that's **0.043 bp per maker leg**.

In [6]:
QQQ_PX = float(rth['close'].iloc[-1])
rebate_bp_per_leg = 0.0030 / QQQ_PX * 1e4
print(f'QQQ ref px: ${QQQ_PX:.2f}   rebate: {rebate_bp_per_leg:.4f} bp/leg')

filled['pnl_rebate_log'] = filled['pnl_net_log'] + rebate_bp_per_leg / 1e4
s_limit_rebate = annualize_sharpe(filled['pnl_rebate_log'], len(filled), sessions_filled)
print(f'Sharpe (limit, post-toxicity + maker rebate): {s_limit_rebate:+.3f}')

QQQ ref px: $694.93   rebate: 0.0432 bp/leg
Sharpe (limit, post-toxicity + maker rebate): -1.114


### The deflation arc, closed

| Stage | Sharpe | Notes |
|---|---:|---|
| Ch8 vectorized in-sample (no costs, snooped) | +2.08 | Ch8 §6 |
| Ch11 walk-forward OOS (no costs) | **+0.88** | Ch11 §7; CI = (−1.6, +3.5) |
| Ch12 cost-aware k\*, market orders, after-cost | **−2.97** | Ch12 §6 |
| **Ch13 passive entry, filled-only, pre-toxicity, pre-cost** | **+0.58** | Apparent rescue |
| **Ch13 passive entry, filled-only, post-toxicity (net exit + commission)** | **−1.20** | Actual realized |
| **Ch13 passive entry, filled-only, post-toxicity + maker rebate** | **−1.11** | Final |

**The execution arc:** +0.88 → −2.97 → −1.11. Passive execution **narrows the loss by ~1.9 Sharpe points** vs market orders — a real improvement worth quantifying. But it does not cross zero. The exit leg still crosses, and the 8.1 bp unfilled-drift means the strategy's biggest winners disappear from the filled ledger.

**Decision rule from the arc:** rebates and passive execution matter when the strategy already has slim-positive expectancy on its own. They don't rescue a structurally cost-dominated strategy.

## §7 — So what?

1. **Passive execution is a tool, not a cure.** Helps when (a) the strategy is cost-dominated, (b) signal has low adverse-selection sensitivity, *and* (c) both legs can be passive. Ch7 MR has (a) and (b) on QQQ but cannot satisfy (c) with a time-stop.
2. **Diagnose toxicity before celebrating the spread credit.**
3. **Watch the unfilled bucket.** Strict toxicity (0.17 bp) is one read; the unfilled-signal drift (+8.1 bp) is often larger.
4. **Match latency to bar resolution.** 1-min/100ms is fine; 1-sec/200ms is not.
5. **Rebates are a tiebreaker, not a strategy.**

## Key Terms

| Term | Definition |
|---|---|
| Market order | Immediate execution at the best available price; taker; pays the spread. |
| Limit order | Execute only at price *p* or better; maker (if posted away from the touch); conditional fill. |
| IOC | Fill what's available now and cancel any rest; prevents leakage. |
| FOK | Fill in full or cancel entirely. |
| Maker | Liquidity provider; earns the maker rebate. |
| Taker | Liquidity remover; pays the taker fee. |
| PFOF | Routing of retail orders to wholesalers for payment; enables zero-commission retail. |
| Fill rate | Filled passive trades / total signals. |
| Adverse selection / toxicity | `drift_unconditional − drift_filled`; how much worse passive fills are than the full signal universe. |
| Latency | Signal-to-fill wall-clock time; random-walk drift = σ<sub>1min</sub> · √(Δt/60). |
| Maker rebate | Per-share credit paid to the maker. NASDAQ tier-1 ≈ $0.0030/share ≈ 0.043 bp/leg on QQQ. |

## Up next

**Ch14 — Position sizing and risk of ruin.** With the after-execution expectancy in hand (here: negative), Kelly tells us how much capital to deploy per trade. Using the pre-cost Sharpe of +0.88 would have us deploying meaningful capital on a strategy that loses money in practice. The Ch12-Ch13 deflation chain is the input that makes Ch14 honest.

## Exercises

1. **Fill-rule strictness.** Re-run the §3 backtest under three fill rules:
   (a) any later bar within K touches the limit (used in the chapter),
   (b) only the *next bar's open* may touch,
   (c) a bar must *trade through* the limit by ε ticks ($0.01 say).
   Compute fill rate, drift_filled, and toxicity under each. Does the +0.17 bp toxicity result survive rule (c)? Which rule is the most realistic for a retail simulator, and why?

2. **Latency sensitivity sweep.** For QQQ at 1-min, 15-sec (resampled), and 5-sec resolutions, compute σ<sub>drift</sub> and E\|drift\| under 100 ms and 500 ms latency assumptions. At what bar resolution does the latency drift exceed the 0.72 bp half-spread credit? *Hint:* you don't need new data; σ<sub>1min</sub> and the √(Δt/60) formula are sufficient.

3. **Maker-rebate breakeven.** Hold fill rate (0.89) and toxicity (0.17 bp) constant. What per-trade gross PnL (in bp) would the strategy need before the 0.043 bp/leg maker rebate is the *deciding* factor between losing and breaking even on after-cost Sharpe? Express the answer in terms of fill rate, gross per-trade std, and the exit-leg cost stack from Ch12.